<a href="https://colab.research.google.com/github/tskir/london-housing/blob/main/notebooks/intended_vs_actual_commencement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Planning London Datahub — intended vs. actual commencement

Scope: **approved** applications with `valid_date` in 2023–2025 and `application_details.intended_commencement_date` on or before 2026-12-31 (capped at 100,000 rows).

For that scope:

1. How many have a **null** `actual_commencement_date` (i.e. haven't reported a start yet)?
2. For the rest, what's the distribution of `actual_commencement_date - intended_commencement_date`?


In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)


## Config

In [ ]:
API_URL = "https://planningdata.london.gov.uk/api-guest"
INDEX = "applications"
HEADERS = {
    "X-API-AllowRequest": "be2rmRnt&",
    "Content-Type": "application/json",
}

PAGE_SIZE = 1000       # docs per scroll page
MAX_RECORDS = 100_000  # cap on the total number of rows pulled

FILTER_QUERY = {
    "bool": {
        "must": [
            {"term": {"status.raw": "Approved"}},
            {
                "range": {
                    "valid_date": {
                        "gte": "01/01/2023",
                        "lte": "31/12/2025",
                    }
                }
            },
            {
                "range": {
                    "application_details.intended_commencement_date": {
                        "lte": "31/12/2026",
                    }
                }
            },
        ]
    }
}

# The intended_commencement_date range filter above already restricts to rows where that field is
# present (a missing field can't match a range query), so every pulled row has an intended date.
SOURCE_FIELDS = [
    "id",
    "lpa_name",
    "status",
    "valid_date",
    "actual_commencement_date",
    "application_details.intended_commencement_date",
]


## Pull the data (scroll API)

In [ ]:
def fetch_subset(index, query, source_fields, page_size, max_records, scroll_ttl="2m"):
    all_hits = []

    init_body = {
        "size": page_size,
        "query": query,
        "_source": source_fields,
    }
    resp = requests.post(
        f"{API_URL}/{index}/_search?scroll={scroll_ttl}",
        headers=HEADERS,
        json=init_body,
    )
    resp.raise_for_status()
    data = resp.json()

    total = data["hits"]["total"]["value"] if isinstance(data["hits"]["total"], dict) else data["hits"]["total"]
    print(f"Matching records available: {total:,}")

    scroll_id = data.get("_scroll_id")
    hits = data["hits"]["hits"]

    while hits:
        all_hits.extend(hits)
        if len(all_hits) >= max_records:
            all_hits = all_hits[:max_records]
            break
        resp = requests.post(
            f"{API_URL}/_search/scroll",
            headers=HEADERS,
            json={"scroll": scroll_ttl, "scroll_id": scroll_id},
        )
        resp.raise_for_status()
        data = resp.json()
        scroll_id = data.get("_scroll_id")
        hits = data["hits"]["hits"]

    print(f"Pulled {len(all_hits):,} records")
    return all_hits


raw_hits = fetch_subset(INDEX, FILTER_QUERY, SOURCE_FIELDS, PAGE_SIZE, MAX_RECORDS)


## Flatten and parse dates

In [ ]:
sources = [h["_source"] for h in raw_hits]
df = pd.json_normalize(sources, sep=".")

intended_col = "application_details.intended_commencement_date"
actual_col = "actual_commencement_date"

for col in [intended_col, actual_col]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format="%d/%m/%Y", errors="coerce")

print(df.shape)
df.head()


## How many have no actual commencement date yet?

In [ ]:
n_total = len(df)
n_null_actual = df[actual_col].isna().sum()
n_present = n_total - n_null_actual

print(f"Approved, valid_date 2023-2025, intended_commencement_date <= 2026: {n_total:,} rows")
print(f"  actual_commencement_date is NULL:    {n_null_actual:,} ({n_null_actual / n_total * 100:.1f}%)")
print(f"  actual_commencement_date is present: {n_present:,} ({n_present / n_total * 100:.1f}%)")


## Distribution of actual − intended commencement date

Only for rows where `actual_commencement_date` is present. Positive values mean the site started later than planned; negative values mean it started earlier than the intended date.

In [ ]:
present = df[df[actual_col].notna()].copy()
present["diff_days"] = (present[actual_col] - present[intended_col]).dt.days

display(present["diff_days"].describe())

n_late = (present["diff_days"] > 0).sum()
n_early = (present["diff_days"] < 0).sum()
n_on_time = (present["diff_days"] == 0).sum()
print(f"Started later than planned:  {n_late:,} ({n_late / n_present * 100:.1f}%)")
print(f"Started earlier than planned: {n_early:,} ({n_early / n_present * 100:.1f}%)")
print(f"Started exactly on the planned date: {n_on_time:,} ({n_on_time / n_present * 100:.1f}%)")


In [ ]:
ax = present["diff_days"].plot(
    kind="hist", bins=60, figsize=(10, 4),
    title=f"actual_commencement_date - intended_commencement_date, in days (n={n_present:,})",
)
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("days (positive = later than planned)")
ax.figure.tight_layout()
plt.show()
